In [ ]:
!pip install agno google-generativeai google-genai

In [ ]:
!pip install -U ddgs newspaper4k lxml_html_clean

In [ ]:
pip install tavily-python

In [ ]:
import os
from agno.agent import Agent
from agno.models.google import Gemini
from typing import List
from agno.team import Team
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.hackernews import HackerNewsTools
from agno.tools.newspaper4k import Newspaper4kTools
from pydantic import BaseModel
from agno.tools.tavily import TavilyTools

os.environ["GOOGLE_API_KEY"] = ""
os.environ["TAVILY_API_KEY"] = ""

class Article(BaseModel):
    title: str
    summary: str
    reference_links: List[str]

In [ ]:
web_searcher = Agent(
    name="Веб-поисковик",
    model=Gemini(id="gemini-2.5-flash-lite"),
    role="Поиск информации по теме в интернете",
    tools=[TavilyTools()],
    add_datetime_to_context=True,
)

redactor = Agent(
    name="Редактор",
    model=Gemini(id="gemini-2.5-flash-lite"),
    role="Ты профессиональный редактор. Твоя задача — написать небольшую, но структурированную и увлекательную статью на основе предоставленных тебе данных.",
    instructions=[
        "Пиши ТОЛЬКО на русском языке.",
        "Структура статьи должна быть такой: заголовок, введение, несколько абзацев с подзаголовками и заключение.",
        "Не ищи информацию в интернете. Работай строго с теми данными, что я тебе дал."
    ],
)

In [ ]:
hn_team = Team(
    name="Команда Писателей",
    model=Gemini(id="gemini-2.5-flash-lite"),
    members=[web_searcher, redactor],
    instructions=[
        "Сначала 'Веб-поисковик' должен найти в интернете информацию по запросу пользователя. "
        "Он должен собрать основные тезисы и, что очень важно, СОХРАНИТЬ ВСЕ ССЫЛКИ на источники.",
        "Затем ты должен передать полученные от веб-поисковика данные (тезисы и ссылки) агенту 'Редактор'. "
        "Его задача — написать на их основе статью.",
        "Покажи пользователю окончательный текст статьи от Редактора.",
    ],
    markdown=True,
    show_members_responses=True,
)

In [ ]:
hn_team.print_response("Три закона робототехники", stream=True)

Output()